# 네이버 API에서 키워드를 입력받아 블로그, 뉴스, 책 데이터 수집
# 각각의 카테고리별로 csv 파일로 저장
# 파일명에는 수집날짜가 자동으로 파일명에 포함되도록
# 반복되는 부분은 함수를 이용해서 중복 줄이기

In [21]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import re
from datetime import datetime
load_dotenv(dotenv_path="./data/.env_db")

True

In [ ]:
def text_clean(text):
    # html 태그를 없애는 정규표현식
    result = re.sub(r"</?[^>]+>", "", text)
    # 한글, 영문, 숫자 외의 모든 문자 제거 후 공백으로 변환
    result = re.sub(r"[^가-힣a-zA-Z0-9]", " ", result)
    result = result.replace("  ", " ").replace("  ", " ").replace("  ", " ")
    return result

# 1페이지 수집

In [34]:
keyword = input("검색할 키워드를 입력해 주세요")

검색할 키워드를 입력해 주세요부동산


In [35]:
user_id = os.getenv("user_id")
user_secret = os.getenv("user_secret")
url = f"https://openapi.naver.com/v1/search/news"
payload = dict(query=keyword, display=100, start=1, sort="date")
headers = {"X-Naver-Client-Id" : user_id, "X-Naver-Client-Secret" : user_secret}
r = requests.get(url, params=payload, headers=headers)
response = r.json()

total_page = response['total'] // 100 + 1

result = {}
for item in response['items']:
#     print(item)
    for key, value in item.items():
        if key in ['title', 'description']:
            result.setdefault(key, []).append(text_clean(value))
        else:
            result.setdefault(key, []).append(value)
df = pd.DataFrame(result)
df

,title,originallink,link,description,pubDate
0,정부 주택시장 안정화 대책 발표 수도권 규제지역 내 15억 초과 주,https://idsn.co.kr/news/view/1065587062100578,https://idsn.co.kr/news/view/1065587062100578,정부가 부동산 세제 조정 과정에서 세제 부담 인상 가능성도 열어뒀다 국토교통부와 기...,"Wed, 15 Oct 2025 13:38:00 +0900"
1,10 15 대책 어떻게 적용되나 일문일답 1주택자 전세대출도 DSR 반영,https://www.newsquest.co.kr/news/articleView.h...,https://www.newsquest.co.kr/news/articleView.h...,금융위원회는 15일 정부서울청사에서 관계부처 합동으로 이같은 내용을 담은 부동산시장...,"Wed, 15 Oct 2025 13:38:00 +0900"
2,배현진 quot 조국도 사는 강남 3구 집값 올린 건 문재인 박원순 커플 quot,https://www.daejonilbo.com/news/articleView.ht...,https://n.news.naver.com/mnews/article/656/000...,배현진 국민의힘 의원은 조국 조국혁신당 비상대책위원장이 오세훈 서울시장의 부동산 정...,"Wed, 15 Oct 2025 13:38:00 +0900"
3,한국인 납치살해 배후 태자그룹 천즈 가 일궈낸 범죄왕국,https://news.jtbc.co.kr/article/NB12266689?inf...,https://n.news.naver.com/mnews/article/437/000...,이후 막대한 자금을 토대로 부동산과 금융업으로 문어발 확장에 나섰고 손을 대는 사업...,"Wed, 15 Oct 2025 13:38:00 +0900"
4,신한투자증권 신한 Premier MyPB 멤버스 관리자산 1조 돌파,https://www.startuptoday.co.kr/news/articleVie...,https://www.startuptoday.co.kr/news/articleVie...,또한 투자 상담 외에도 세무 부동산 은퇴설계 등 다양한 전문 콘텐츠와 프리미엄 혜택...,"Wed, 15 Oct 2025 13:38:00 +0900"
...,...,...,...,...,...
95,뉴스나우 이재명 정부 세 번째 부동산 대책 집값 잡을까,https://www.ytn.co.kr/_ln/0102_202510151313199176,https://n.news.naver.com/mnews/article/052/000...,부동산 경제연구소장 아래 텍스트는 실제 방송 내용과 차이가 있을 수 있으니 보다 정...,"Wed, 15 Oct 2025 13:13:00 +0900"
96,10 15 부동산 대책 발표 서울 경기 12곳 규제지역 설정,https://www.news1.kr/photos/7542411,https://n.news.naver.com/mnews/article/421/000...,15일 경기 광주시 남한산성에서 내려다본 서울의 아파트 단지 모습 이날 정부는 기존...,"Wed, 15 Oct 2025 13:13:00 +0900"
97,정부 10 15 부동산 대책 발표 서울전역 경기 12곳 묶었다,https://www.news1.kr/photos/7542414,https://n.news.naver.com/mnews/article/421/000...,15일 경기 광주시 남한산성에서 내려다본 서울의 아파트 단지 모습 이날 정부는 기존...,"Wed, 15 Oct 2025 13:13:00 +0900"
98,10 15 부동산 대책 발표 서울 경기 12곳 규제지역 설정 주담대 한,https://www.news1.kr/photos/7542409,https://n.news.naver.com/mnews/article/421/000...,15일 경기 광주시 남한산성에서 내려다본 서울의 아파트 단지 모습 이날 정부는 기존...,"Wed, 15 Oct 2025 13:13:00 +0900"


# 전체페이지 수집

In [44]:
keyword = input("검색할 키워드를 입력해 주세요")


blog_data = []
news_data = []
book_data = []

category = ['blog', 'news', 'book']

for cate in category:
    page_num = 1
    total_page = 1
    start_num = 1
    while page_num <= total_page:
        print(cate, page_num, total_page, end="\r")
        user_id = os.getenv("user_id")
        user_secret = os.getenv("user_secret")
        url = f"https://openapi.naver.com/v1/search/{cate}"
        payload = dict(query=keyword, display=100, start=start_num, sort="date")
        headers = {"X-Naver-Client-Id" : user_id, "X-Naver-Client-Secret" : user_secret}
        r = requests.get(url, params=payload, headers=headers)
        response = r.json()

        result = {}
        for item in response['items']:

            for key, value in item.items():
                if key in ['title', 'description']:
                    result.setdefault(key, []).append(text_clean(value))
                else:
                    result.setdefault(key, []).append(value)
        
        if cate == 'blog':
            blog_data.append(pd.DataFrame(result))
        elif cate == 'news':
            news_data.append(pd.DataFrame(result))
        elif cate == 'book':
            book_data.append(pd.DataFrame(result))

        total_page = response['total'] // 100 + 1
        if total_page > 11:
            total_page = 11
        else:
            total_page = response['total'] // 100 + 1


        page_num += 1

        if start_num < 901:
            start_num += 100
        elif start_num > 900:
            start_num += 99

            
            
blog_df = pd.concat(blog_data)
news_df = pd.concat(news_data)
book_df = pd.concat(book_data)

blog_df = blog_df.reset_index(drop=True)
news_df = news_df.reset_index(drop=True)
book_df = book_df.reset_index(drop=True)

display(blog_df)
display(news_df)
display(book_df)

검색할 키워드를 입력해 주세요부동산


,title,link,description,bloggername,bloggerlink,postdate
0,배방읍 배방푸르지오2차 206동 매매 2억 8 500,https://blog.naver.com/5422249/224041998993,부동산 거래에 있어서 전문적이고 신뢰할 수 있는 고객님의 파트너 입니다 해당포스팅은...,우방탑부동산,blog.naver.com/5422249,20251015
1,대구파이썬크롤링 데이터 시대의 핵심 기술을 내 것으로 만들자,https://blog.naver.com/ajswl1255/224041999102,대구파이썬크롤링 배우면 할 수 있는 것들 원하는 쇼핑몰 상품 가격 자동 수집 및 비...,한번의 클릭이 도움이 되요!,blog.naver.com/ajswl1255,20251015
2,부천 로펌에서 자주 발생하는 사건 유형,https://blog.naver.com/ogenu/224041998423,부동산 사건 부동산 거래와 관련된 사건은 부천 지역에서도 빈번하게 발생합니다 계약 ...,ogenu님의 블로그,blog.naver.com/ogenu,20251015
3,이준석 부동산 정책으로 정부 2 0 선언 공급은,https://blog.naver.com/elodialopez/224041997888,연합 헤럴드경제 김진 기자 이준석 개혁신당 대표는 15일 부동산 정책으로 이재명 ...,오늘보다 더 나은 내일을 약속합니다,blog.naver.com/elodialopez,20251015
4,인천전세사기변호사 법적 조언은,https://blog.naver.com/pns4042/224041998470,마지막으로 주변의 추천이나 후기 등을 통해 신뢰할 수 있는 부동산 중개업체를 이용하...,법무법인일신강남분사무소,blog.naver.com/pns4042,20251015
...,...,...,...,...,...,...
1095,평택시 서탄면 내천리 건물매매 대리점 본사,https://blog.naver.com/kims114cc/224041932992,3 환산 만원 부동산 031 349 1000 부동산천왕 공인중개사 매물 상세주소 h...,토지 전문 부동산 010-5281-9938,blog.naver.com/kims114cc,20251015
1096,도원동 도원삼성래미안 아파트 112동 108A 84 전세 고 22층,https://blog.naver.com/koreana3316/224041932958,해당포스팅은 네이버부동산에서 소유자가 검증된 확인매물 입니다 2555371672 도...,코리아나공인중개사,blog.naver.com/koreana3316,20251015
1097,10 15 대출 규제 핵심 완벽 정리 나에게 미치는 영향은,https://blog.naver.com/eelion/224041932193,10 15 부동산 대책 10월 16일부터 즉시 시행되는 새로운 대출 규제 총정리 ...,불당홈즈공인중개사,blog.naver.com/eelion,20251015
1098,행정심판청구서 예시 담배소매인지정불가처분취소심판청구,https://blog.naver.com/rewaterguy/224041932802,담배소매인지정불가처분취소심판청구 청 구 취 지 피청구인이 2014 12 22 청구인...,김재호변호사,blog.naver.com/rewaterguy,20251015


,title,originallink,link,description,pubDate
0,미국 영국도 캄보디아 범죄조직 정조준 코인 압류 기소,http://www.yonhapnewstv.co.kr/MYH2025101513481...,https://n.news.naver.com/mnews/article/422/000...,영국 정부 역시 천즈 회장과 그 측근 등이 소유한 런던의 230억원 짜리 고급 저택...,"Wed, 15 Oct 2025 13:49:00 +0900"
1,아유경제 부동산 성남시 수도권 충청권 연결 중부권 광역급행철도,http://www.areyou.co.kr/news/articleView.html?...,http://www.areyou.co.kr/news/articleView.html?...,경기 성남시 등 7개 지자체가 수도권과 충청권을 연결하는 광역급행철도의 조속한 착공...,"Wed, 15 Oct 2025 13:48:00 +0900"
2,신세계프라퍼티 GRESB 평가 2년 연속 최고 등급 quot 지속가능개발 선두,https://news.mtn.co.kr/news-detail/20251015134...,https://news.mtn.co.kr/news-detail/20251015134...,GRESB는 부동산 실물 자산과 운용사를 대상으로 환경 및 사회에 미치는 영향과 이...,"Wed, 15 Oct 2025 13:48:00 +0900"
3,신한투자증권 신한 Premier MyPB 멤버스 관리자산 1조원 돌파,http://www.biztribune.co.kr/news/articleView.h...,http://www.biztribune.co.kr/news/articleView.h...,투자 상담뿐 아니라 세무 부동산 은퇴 설계 등 다양한 전문 콘텐츠와 프리미엄 혜택을...,"Wed, 15 Oct 2025 13:48:00 +0900"
4,정부 부동산 세제 개편 시사 quot 보유세 강화 거래세 조정 검토 quot,https://www.newswhoplus.com/news/articleView.h...,https://www.newswhoplus.com/news/articleView.h...,정부가 토지거래허가구역 확대와 고가주택 대출 규제 강화에 이어 부동산 세제 조정 가...,"Wed, 15 Oct 2025 13:48:00 +0900"
...,...,...,...,...,...
1095,서울 전역 경기 12곳 규제지역 확대 집값 확산에 강수,https://www.chosun.com/economy/real_estate/202...,https://n.news.naver.com/mnews/article/023/000...,정부가 부동산 시장 안정화를 위해 서울 전역과 경기도 일부 지역을 규제 지역으로 지...,"Wed, 15 Oct 2025 10:01:00 +0900"
1096,quot 고가 아파트 자금출처 전방위 검증 부동산 탈세 신고센터 설치 quot,http://www.fnnews.com/news/202510150916434461,https://n.news.naver.com/mnews/article/014/000...,임 장관은 이날 정부서울청사에서 열린 부동산관계장관회의 모두발언을 통해 quot 서...,"Wed, 15 Oct 2025 10:01:00 +0900"
1097,속보 서울 전역 경기 12곳 규제지역 토허구역 묶는다,https://www.hankyung.com/article/202510159416i,https://n.news.naver.com/mnews/article/015/000...,여기에 부동산 세제 개편 계획까지 제시하면서 사실상 정부가 동원할 수 있는 모든 부...,"Wed, 15 Oct 2025 10:01:00 +0900"
1098,서울 경기 12곳 토지거래허가구역 지정 3중규제로 묶는다 10 15 부동,http://www.fnnews.com/news/202510150753317335,https://n.news.naver.com/mnews/article/014/000...,열린 부동산 관계장관회의에서 주택시장 안정화 대책 을 발표했다 주택수요 관리 부동산...,"Wed, 15 Oct 2025 10:01:00 +0900"


,title,link,image,author,discount,publisher,pubdate,isbn,description
0,2026 경록 주택관리사 기본서 1차 회계원리,https://search.shopping.naver.com/book/catalog...,https://shopping-phinf.pstatic.net/main_565726...,경록 주택관리사 교재편찬위원회^신한부동산연구소,37800,경록,20260105,9791194560326,경록 100 합격프로젝트 1 경록 교재 시리즈의 특징 1 경록교재는 누구나 신문을 ...
1,2026 경록 주택관리사 회차별 기출문제 1차과목,https://search.shopping.naver.com/book/catalog...,https://shopping-phinf.pstatic.net/main_565726...,경록 주택관리사 교재편찬위원회^신한부동산연구소,26100,경록,20260105,9791194560296,경록 또 100 적중 노리는 주택관리사 기출문제집 출간되었다 첫째 이 책의 출판 형...
2,2026 경록 주택관리사 기본서 1차 민법,https://search.shopping.naver.com/book/catalog...,https://shopping-phinf.pstatic.net/main_565726...,경록 주택관리사 교재편찬위원회^신한부동산연구소,39600,경록,20260105,9791194560340,경록 100 합격프로젝트 1 경록교재 시리즈의 특징 1 경록교재는 누구나 신문을 읽...
3,2026 경록 주택관리사 기본서 1차 공동주택시설개론,https://search.shopping.naver.com/book/catalog...,https://shopping-phinf.pstatic.net/main_565726...,경록 주택관리사 교재편찬위원회^신한부동산연구소,38700,경록,20260105,9791194560333,경록 100 합격프로젝트 1 경록교재 시리즈의 특징 1 경록교재는 누구나 신문을 읽...
4,2026 경록 주택관리사 기본서 1차과목세트,https://search.shopping.naver.com/book/catalog...,https://shopping-phinf.pstatic.net/main_565726...,경록 주택관리사 교재편찬위원회^신한부동산연구소,116100,경록,20260105,9791194560319,경록 100 합격프로젝트 1 경록교재 시리즈의 특징 1 경록교재는 누구나 신문을 읽...
...,...,...,...,...,...,...,...,...,...
1095,2024 에듀윌 공인중개사 2차 기본서 부동산공법 합격자 수가 선택의 기준,https://search.shopping.naver.com/book/catalog...,https://shopping-phinf.pstatic.net/main_450077...,오시훈,38700,에듀윌,20240107,9791136099747,제 35회 공인중개사 시험 합격의 바이블 10개년 시험 기출 빅데이터 분석 35회 ...
1096,2024 에듀윌 공인중개사 2차 기본서 부동산공시법 합격자수가 선택의 기준,https://search.shopping.naver.com/book/catalog...,https://shopping-phinf.pstatic.net/main_450082...,김민석,36900,에듀윌,20240107,9791136099730,제 35회 공인중개사 시험 합격의 바이블 10개년 시험 기출 빅데이터 분석 35회 ...
1097,2024 경록 공인중개사 핵심요약집 민법 및 민사특별법,https://search.shopping.naver.com/book/catalog...,https://shopping-phinf.pstatic.net/main_426178...,경록 공인중개사 교재편찬위원회^신한부동산연구소,24300,경록,20240105,9791192336909,대한민국에서 유일하게 부동산 전문교육 67년의 전통과 노하우 그리고 최고 최대출제위...
1098,2024 경록 공인중개사 핵심요약집 2차 부동산공법,https://search.shopping.naver.com/book/catalog...,https://shopping-phinf.pstatic.net/main_426178...,경록 공인중개사 교재편찬위원회^신한부동산연구소,24300,경록,20240105,9791192336930,경록은 대한민국에서 유일하게 부동산 전문교육 67년의 전통과 노하우 그리고 최고 최...


In [36]:
news_data

[]

In [40]:
news_df = pd.concat(news_data)

In [41]:
news_df

,title,originallink,link,description,pubDate
0,서울 전역 경기 12곳 규제지역 토허구역 대출 묶는다,http://www.yonhapnewstv.co.kr/MYH2025101513423...,https://n.news.naver.com/mnews/article/422/000...,배시진 기자 기자 네 오늘 15일 오전 이재명 정부가 세 번째 부동산 대책을 발표했...,"Wed, 15 Oct 2025 13:43:00 +0900"
1,로펌 브리핑 법무법인 세종 건설클레임센터 신설,https://www.dnews.co.kr/uhtml/view.jsp?idxno=2...,https://www.dnews.co.kr/uhtml/view.jsp?idxno=2...,사업과 부동산 개발산업 공공 민간 건설 프로젝트 수행 과정에서 발생하는 다양한 건설...,"Wed, 15 Oct 2025 13:42:00 +0900"
2,대출한도 또 축소 고가주택 갈아타기 수요 누른다,https://www.goodnews1.com/news/articleView.htm...,https://www.goodnews1.com/news/articleView.htm...,고가주택 집값 상승세가 전체 부동산 시장 과열을 이끌고 있기 때문에 초고가 주택 수...,"Wed, 15 Oct 2025 13:42:00 +0900"
3,10 15 대책 연소득 1억 차주 대출한도 최대 14 7 줄어든다 종합,https://news.einfomax.co.kr/news/articleView.h...,https://news.einfomax.co.kr/news/articleView.h...,금융위원회가 발표한 부동산시장 안정 대책에 따르면 16일부터 수도권 규제지역의 시가...,"Wed, 15 Oct 2025 13:42:00 +0900"
4,우리은행 홈체크와 제휴 아파트 하자 점검 서비스 제공,http://www.newscape.co.kr/news/articleView.htm...,http://www.newscape.co.kr/news/articleView.htm...,우리은행 관계자는 quot 이번 제휴를 통해 고객은 부동산과 금융 서비스뿐 아니라 ...,"Wed, 15 Oct 2025 13:42:00 +0900"
...,...,...,...,...,...
95,서울 전역 경기 남부 12곳 규제 토허구역 묶는다,https://www.etnews.com/20251015000045,https://n.news.naver.com/mnews/article/030/000...,국토부는 허위신고 가격 띄우기 행위에 대해 기획조사를 벌이고 금융위와 국세청 경찰청...,"Wed, 15 Oct 2025 10:01:00 +0900"
96,고가주택 주담대 한도 6억 2억 전세대출 DSR 적용,https://www.newsis.com/view/NISX20251015_00033...,https://n.news.naver.com/mnews/article/003/001...,정부는 15일 오전 경제부총리 주재로 국토부장관 금융위원장이 참석하는 부동산관계장관...,"Wed, 15 Oct 2025 10:01:00 +0900"
97,속보 경기 과천 광명 등 12곳 투기과열지구 토허구역 동시 지정,http://www.inews24.com/view/1895469,https://n.news.naver.com/mnews/article/031/000...,이재명 정부의 세번째 부동산 대책에 경기 과천과 광명 등 12개 지자체를 조정대상지...,"Wed, 15 Oct 2025 10:01:00 +0900"
98,내일부터 수도권 25억 초과 주택담보 대출 한도 2억 원으로 축소,https://imnews.imbc.com/news/2025/econo/articl...,https://n.news.naver.com/mnews/article/214/000...,금융위원회는 10 15 부동산 대책의 일환으로 15억 원 이하 주택을 살 땐 대출 ...,"Wed, 15 Oct 2025 10:01:00 +0900"


# 코드를 함수화

In [64]:
def naver_api(keyword, cate):
    page_num = 1
    total_page = 1
    start_num = 1
    
    result = {}
    while page_num <= total_page:
        print(cate, page_num, total_page, end="\r")
        user_id = os.getenv("user_id")
        user_secret = os.getenv("user_secret")
        url = f"https://openapi.naver.com/v1/search/{cate}"
        payload = dict(query=keyword, display=100, start=start_num, sort="date")
        headers = {"X-Naver-Client-Id" : user_id, "X-Naver-Client-Secret" : user_secret}
        r = requests.get(url, params=payload, headers=headers)
        response = r.json()

        
        for item in response['items']:

            for key, value in item.items():
                if key in ['title', 'description']:
                    result.setdefault(key, []).append(text_clean(value))
                else:
                    result.setdefault(key, []).append(value)
        
        
        
        total_page = response['total'] // 100 + 1
        if total_page > 11:
            total_page = 11
        else:
            total_page = response['total'] // 100 + 1


        page_num += 1

        if start_num < 901:
            start_num += 100
        elif start_num > 900:
            start_num += 99

    return pd.DataFrame(result)

In [65]:
today = datetime.today()
today

datetime.datetime(2025, 10, 15, 14, 35, 25, 508407)

In [66]:
date = f"{today.year}{today.month:02d}{today.day:02d}"
date

'20251015'

In [67]:
keyword = input("검색어를 입력하세요")
date = f"{today.year}{today.month:02d}{today.day:02d}"
for cate in category:
    result = naver_api(keyword, cate)
    result.to_csv(f"./data/{keyword}_{cate}_{date}.csv", index=False, encoding="utf-8-sig")

검색어를 입력하세요핀테크
